# 🇩🇿 Fine-Tuning Whisper-Tiny on Algerian Dialects (dialects-ai-foundry)
تدريب وضبط نموذج Whisper على حزمة البيانات المجمعة عبر الولايات الست.

In [ ]:
# 1. تثبيت الحزم المطلوبة
!pip install -q transformers datasets accelerate evaluate jiwer soundfile peft bitsandbytes

In [ ]:
# 2. استنساخ المستودع لجلب البيانات المقسمة وملفات الصوت
!git clone https://github.com/attitawfikatti-alt/dialects-ai-foundry.git
%cd dialects-ai-foundry

In [ ]:
# 3. تحميل ومعاينة البيانات عبر datasets
import json
from datasets import Dataset, Audio

def read_jsonl(path):
    samples = {'audio': [], 'sentence': []}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            samples['audio'].append(f"dataset/audio/{item['id']}.wav")
            samples['sentence'].append(item['normalized_text'])
    return Dataset.from_dict(samples).cast_column('audio', Audio(sampling_rate=16000))

train_ds = read_jsonl('dataset/train.jsonl')
val_ds = read_jsonl('dataset/val.jsonl')
print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

In [ ]:
# 4. إعداد المعالج (Processor) ونموذج Whisper-Tiny
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_id = 'openai/whisper-tiny'
processor = WhisperProcessor.from_pretrained(model_id, language='arabic', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
# 5. معالجة الإشارات الصوتية والنصوص
def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = processor.feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_features[0]
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

train_ds = train_ds.map(prepare_dataset, remove_columns=train_ds.column_names)
val_ds = val_ds.map(prepare_dataset, remove_columns=val_ds.column_names)

In [ ]:
# 6. إعداد مُجمّع البيانات ومقاييس التقييم
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
wer_metric = evaluate.load('wer')

In [ ]:
# 7. تشغيل دورة التدريب السريعة (Fine-Tuning)
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-dz-checkpoints',
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=20,
    max_steps=150,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy='steps',
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    report_to=['none'],
    save_total_limit=1
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor
)

trainer.train()